In [ ]:
import re
import pandas as pd

def parse_time_to_ms(time_str):
    """Converts time strings like '1.32s' or '446.59ms' to a float in milliseconds."""
    if not time_str: return None
    if time_str.endswith('ms'):
        return float(time_str.replace('ms', ''))
    elif time_str.endswith('s'):
        return float(time_str.replace('s', '')) * 1000
    return float(time_str)

def extract_benchmark_data(file_content):
    # Split the document by the thread headers
    sections = re.split(r'## (\d+) threads', file_content)[1:]

    results = []

    # Process in pairs: (thread_count, section_content)
    for i in range(0, len(sections), 2):
        thread_count = int(sections[i])
        content = sections[i+1]

        # 1. Extract JVM Metrics
        # Look for the CSV header and the data lines following it
        jvm_match = re.search(r'elapsed_seconds.*?\n(.*?)\n```', content, re.DOTALL)
        avg_cpu = 0
        avg_stack = 0
        if jvm_match:
            lines = [line.strip() for line in jvm_match.group(1).strip().split('\n') if line.strip()]
            cpu_sum = 0
            stack_sum = 0
            for line in lines:
                cols = [c.strip() for c in line.split(',')]
                cpu_sum += float(cols[1])
                stack_sum += float(cols[3])

            avg_cpu = cpu_sum / len(lines) if lines else 0
            avg_stack = stack_sum / len(lines) if lines else 0

        # 2. Extract wrk Metrics
        req_sec = re.search(r'Requests/sec:\s+([\d\.]+)', content)
        avg_lat = re.search(r'Latency\s+([\d\.]+(?:ms|s))', content)
        p50_lat = re.search(r'50%\s+([\d\.]+(?:ms|s))', content)
        p99_lat = re.search(r'99%\s+([\d\.]+(?:ms|s))', content)

        results.append({
            'Thread Count': thread_count,
            'Requests/sec': float(req_sec.group(1)) if req_sec else None,
            'Avg Latency (ms)': parse_time_to_ms(avg_lat.group(1)) if avg_lat else None,
            'P50 Latency (ms)': parse_time_to_ms(p50_lat.group(1)) if p50_lat else None,
            'P99 Latency (ms)': parse_time_to_ms(p99_lat.group(1)) if p99_lat else None,
            'Avg CPU (%)': round(avg_cpu, 2),
            'Avg Stack (MB)': round(avg_stack, 2)
        })

    # Build DataFrame
    df = pd.DataFrame(results)
    df = df.sort_values(by='Thread Count', ascending=True).reset_index(drop=True)
    return df

# Load the file (assuming it's saved locally)
with open('thread-pool/determining-optimal-thread-count.md', 'r') as f:
    content = f.read()

df = extract_benchmark_data(content)

# Print DataFrame for verification
print("--- Extracted DataFrame ---")
print(df.to_string(index=False))

# Generate LaTeX table formatting (utilizing booktabs)
latex_table = df.to_latex(
    index=False,
    float_format="%.2f",
    column_format="ccccccc",
    caption="Performance metrics across varying thread pool sizes at 10,000 connections",
    label="tab:thread_pool_opt",
    position="!htbp",
    escape=False
)

# print("\n--- Generated LaTeX Table ---")
# print(latex_table)

In [ ]:
import matplotlib.pyplot as plt

# Ensure the DataFrame is sorted by Thread Count to sequence the staggering correctly
df = df.sort_values(by='Thread Count').reset_index(drop=True)

# Increased figure width slightly to give labels more breathing room
fig, ax1 = plt.subplots(figsize=(12, 7))

# Plot Requests/sec on the primary y-axis (left)
color1 = 'tab:blue'
ax1.set_xlabel('Thread count', fontsize=12)
ax1.set_ylabel('Requests/sec', color=color1, fontsize=12)
line1 = ax1.plot(df['Thread Count'], df['Requests/sec'], color=color1, marker='o', label='Requests/sec')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, linestyle='--', alpha=0.6)

# Create a secondary axis sharing the same x-axis
ax2 = ax1.twinx()

# Plot Avg CPU (%) on the secondary y-axis (right)
color2 = 'tab:red'
ax2.set_ylabel('Avg CPU (%)', color=color2, fontsize=12)
line2 = ax2.plot(df['Thread Count'], df['Avg CPU (%)'], color=color2, marker='s', linestyle='--', label='Avg CPU (%)')
ax2.tick_params(axis='y', labelcolor=color2)

# Define 4 distinct vertical offset levels (in points) to prevent clustering
height_levels = [15, 35, 55, 75]

# Annotate points with X-values (Thread count) only
for i, row in df.iterrows():
    # Cycle through the 4 height levels based on the data index
    y_offset = height_levels[i % 4]

    # Annotate the Thread count above the Requests/sec line
    ax1.annotate(f"{int(row['Thread Count'])}",
                 (row['Thread Count'], row['Requests/sec']),
                 textcoords="offset points",
                 xytext=(0, y_offset),
                 ha='center', fontsize=9, color='black',
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8),
                 arrowprops=dict(arrowstyle="-", color='gray', alpha=0.6))

# Combine legends from both axes
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='lower right')

# --- Title Updates (Tightened Spacing) ---
# Lowered the suptitle slightly using the y parameter (default is close to 1.0)
fig.suptitle('Thread pool optimization: Throughput vs. CPU overhead',
             fontsize=16, fontweight='bold', y=0.97)

# Reduced pad from 20 to 10 to bring the subtitle closer to the plot box
ax1.set_title('(test conditions: 10,000 concurrency, 100 ms delay, 4 vCPUs)',
              fontsize=12, fontweight='normal', pad=10)

# Adjusted the layout rectangle to push the plot slightly up towards the titles
fig.tight_layout(rect=[0, 0, 1, 0.96])

# Save the plot with bbox_inches='tight' to ensure titles are not cut off
plt.savefig('thread-pool-opt.png', dpi=300, bbox_inches='tight')

# Display the plot
plt.show()